In [1]:
# Import the Path class to work with file system paths in a
# platform-independent and readable way.
from pathlib import Path

In [2]:
# Kaggle automatically mounts all attached datasets under
# the /kaggle/input directory.
#
# Before processing any files, verify that the dataset has
# been mounted correctly and identify its folder name.
for folder in Path("/kaggle/input").iterdir():
    print(folder)

/kaggle/input/datasets


In [3]:
# If the dataset is organized into subdirectories, inspect the
# next level to locate the folder containing the IRS PDF files.
for folder in Path("/kaggle/input/datasets").iterdir():
    print(folder)

/kaggle/input/datasets/nallagantisharath


In [4]:
# Define the location of the dataset containing the IRS
# documents. Update this path if the dataset name changes.
INPUT_DIR = Path("/kaggle/input/datasets/nallagantisharath")

# Define the directory where all extracted machine-readable
# data will be written.
#
# The /kaggle/working directory is writable and can be used
# to store outputs generated during notebook execution.
OUTPUT_DIR = Path("/kaggle/working/irs-extracted-data")

# Create the output directory if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Recursively search the dataset directory for every PDF file.
#
# The rglob() method searches all nested folders, allowing the
# project to support any directory structure.
pdf_files = sorted(INPUT_DIR.rglob("*.pdf"))

# Display the number of PDF documents found.
print(f"Number of PDF files found: {len(pdf_files)}\n")

# Display the full path of every discovered PDF document.
for pdf_file in pdf_files:
    print(pdf_file)

Number of PDF files found: 14

/kaggle/input/datasets/nallagantisharath/1120-returns/Form 1125-A (Cost of Goods Sold).pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/Instructions for Form 4626.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/Instructions for Form 4797.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/Instructions for Schedule M-3.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/SCHEDULE M-3 .pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/Schedule D .pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/f1120.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/f1120sd.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/f1125e.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/f4626.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/f4797.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/f5472.pdf
/kaggle/input/datasets/nallagantisharath/1120-returns/i1120.pdf
/kaggle/input/datasets/nall

In [6]:
# Install PyMuPDF.
#
# PyMuPDF provides fast and accurate PDF text extraction,
# making it well suited for IRS forms and instruction manuals.
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 73.7 MB/s eta 0:00:00


In [7]:
# Import PyMuPDF.
#
# The package is installed as "pymupdf" but imported using
# the module name "fitz".
import fitz

# Import the JSON module for writing extracted records
# to machine-readable JSON files.
import json

In [8]:
def extract_pdf(pdf_path: Path):
    """
    Extract the textual content from every page of a PDF document.

    Each page is stored as an individual record so that page-level
    metadata is preserved throughout the data processing pipeline.

    Parameters
    ----------
    pdf_path : Path
        Path to the PDF document.

    Returns
    -------
    list
        A list of dictionaries containing the document name,
        page number, and extracted text.
    """

    # Open the PDF document.
    document = fitz.open(pdf_path)

    # Store the extracted page records.
    pages = []

    # Iterate through every page in the document.
    for page_number in range(document.page_count):

        # Load the current page.
        page = document.load_page(page_number)

        # Extract the page text.
        text = page.get_text("text")

        # Store the extracted information.
        pages.append(
            {
                "document_name": pdf_path.name,
                "page_number": page_number + 1,
                "text": text,
            }
        )

    # Close the document to release system resources.
    document.close()

    return pages

In [9]:
# Store the extracted pages from every PDF.
all_pages = []

# Process every PDF document found in the dataset.
for pdf in pdf_files:

    print(f"Processing: {pdf.name}")

    # Extract every page from the current PDF.
    pages = extract_pdf(pdf)

    # Add the extracted pages to the master collection.
    all_pages.extend(pages)

# Display the total number of extracted pages.
print(f"\nTotal pages extracted: {len(all_pages)}")

Processing: Form 1125-A (Cost of Goods Sold).pdf
Processing: Instructions for Form 4626.pdf
Processing: Instructions for Form 4797.pdf
Processing: Instructions for Schedule M-3.pdf
Processing: SCHEDULE M-3 .pdf
Processing: Schedule D .pdf
Processing: f1120.pdf
Processing: f1120sd.pdf
Processing: f1125e.pdf
Processing: f4626.pdf
Processing: f4797.pdf
Processing: f5472.pdf
Processing: i1120.pdf
Processing: i5472.pdf

Total pages extracted: 133


In [10]:
# Display information about the first extracted page to verify
# that the extraction process worked correctly.
print("Document:", all_pages[0]["document_name"])
print("Page:", all_pages[0]["page_number"])

print("\nExtracted Text Preview\n")
print("-" * 80)

# Display the first 3,000 characters of the extracted text.
print(all_pages[0]["text"][:3000])

Document: Form 1125-A (Cost of Goods Sold).pdf
Page: 1

Extracted Text Preview

--------------------------------------------------------------------------------
Form 1125-A
(Rev. November 2024)
Department of the Treasury  
Internal Revenue Service 
Cost of Goods Sold
Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065. 
Go to www.irs.gov/Form1125A for the latest information.  
OMB No. 1545-0123
Name
Employer identification number
1
Inventory at beginning of year  .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
1
2
Purchases .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
2
3
Cost of labor 
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
3
4
Additional section 263A costs (attach schedule) .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
4
5
Other costs (attach schedule) 
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
5
6
Total. Add lines 1 through 5 .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
6
7
Inventory at end of year .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.


In [11]:
# Define the output file where all extracted PDF pages will be stored.
#
# JSONL means "JSON Lines." Each line in the file contains one
# complete JSON record representing one page from one PDF document.
RAW_OUTPUT_FILE = OUTPUT_DIR / "irs_pdf_pages_raw.jsonl"

# Open the output file in write mode using UTF-8 encoding.
with RAW_OUTPUT_FILE.open("w", encoding="utf-8") as file:

    # Write each extracted page as a separate JSON object.
    for page_record in all_pages:

        file.write(
            json.dumps(
                page_record,
                ensure_ascii=False,
            )
            + "\n"
        )

# Confirm that the file was created successfully.
print(f"Raw extracted data saved to:\n{RAW_OUTPUT_FILE}")

Raw extracted data saved to:
/kaggle/working/irs-extracted-data/irs_pdf_pages_raw.jsonl


In [12]:
# Confirm that the JSONL file exists.
print("File exists:", RAW_OUTPUT_FILE.exists())

# Display the file size in bytes.
print("File size:", RAW_OUTPUT_FILE.stat().st_size, "bytes")

File exists: True
File size: 760028 bytes


In [13]:
# Read the saved JSONL file to verify that the records were
# written correctly and can be loaded again.
with RAW_OUTPUT_FILE.open("r", encoding="utf-8") as file:

    # Read and display the first three page records.
    for record_number in range(3):

        line = file.readline()

        # Convert the JSON text back into a Python dictionary.
        record = json.loads(line)

        print(
            f"\nRecord {record_number + 1}:",
            record["document_name"],
            "- Page",
            record["page_number"],
        )

        print(record["text"][:500])
        print("-" * 80)


Record 1: Form 1125-A (Cost of Goods Sold).pdf - Page 1
Form 1125-A
(Rev. November 2024)
Department of the Treasury  
Internal Revenue Service 
Cost of Goods Sold
Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065. 
Go to www.irs.gov/Form1125A for the latest information.  
OMB No. 1545-0123
Name
Employer identification number
1
Inventory at beginning of year  .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
1
2
Purchases .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
2
3
Cost of labor 
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

--------------------------------------------------------------------------------

Record 2: Form 1125-A (Cost of Goods Sold).pdf - Page 2
Form 1125-A (Rev. 11-2024) 
Page 2 
Specific Instructions
Line 1. Inventory at Beginning of 
Year
If you are changing your method of 
accounting for the current tax year, you 
must refigure last year’s closing inventory 
using the new method of accounting. Enter 
the result on line 1. If there is a diff

In [14]:
# Identify pages where no readable text was extracted.
#
# Empty pages may be blank, image-based, scanned, or heavily
# dependent on graphical form elements.
empty_pages = [
    page
    for page in all_pages
    if not page["text"].strip()
]

print(f"Number of empty pages: {len(empty_pages)}")

# Display the first few empty-page locations for review.
for page in empty_pages[:20]:
    print(
        page["document_name"],
        "- Page",
        page["page_number"],
    )

Number of empty pages: 0


In [15]:
import re


def clean_pdf_text(raw_text: str) -> str:
    """
    Clean text extracted from an IRS PDF while preserving
    important tax terminology, line numbers, headings,
    percentages, amounts, and legal references.

    Parameters
    ----------
    raw_text : str
        Text extracted directly from a PDF page.

    Returns
    -------
    str
        Cleaned and normalized text.
    """

    # Return an empty string when no text was extracted.
    if not raw_text:
        return ""

    # Normalize Windows and older Mac line endings.
    text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

    # Join words that were split across lines with a hyphen.
    #
    # Example:
    # "corpora-\ntion" becomes "corporation".
    text = re.sub(
        r"([A-Za-z])-\n([A-Za-z])",
        r"\1\2",
        text,
    )

    # Remove spaces or tabs immediately before line breaks.
    text = re.sub(r"[ \t]+\n", "\n", text)

    # Replace multiple spaces and tabs with one space.
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Reduce three or more consecutive blank lines to two.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove unnecessary whitespace from the beginning
    # and end of the page.
    return text.strip()

In [16]:
# Create a new collection for cleaned page records.
#
# The original raw text is preserved so that we can compare
# the cleaned result with the original PDF extraction.
cleaned_pages = []

for page_record in all_pages:

    # Clean the raw text from the current page.
    cleaned_text = clean_pdf_text(
        page_record["text"]
    )

    # Create a new record containing both raw and cleaned text.
    cleaned_record = {
        "document_name": page_record["document_name"],
        "page_number": page_record["page_number"],
        "raw_text": page_record["text"],
        "clean_text": cleaned_text,
        "raw_character_count": len(page_record["text"]),
        "clean_character_count": len(cleaned_text),
    }

    cleaned_pages.append(cleaned_record)

print(f"Number of cleaned page records: {len(cleaned_pages)}")

Number of cleaned page records: 133


In [17]:
# Select one page to compare before and after cleaning.
sample_page = cleaned_pages[0]

print("Document:", sample_page["document_name"])
print("Page:", sample_page["page_number"])

print("\nRAW TEXT")
print("-" * 80)
print(sample_page["raw_text"][:1500])

print("\nCLEANED TEXT")
print("-" * 80)
print(sample_page["clean_text"][:1500])

Document: Form 1125-A (Cost of Goods Sold).pdf
Page: 1

RAW TEXT
--------------------------------------------------------------------------------
Form 1125-A
(Rev. November 2024)
Department of the Treasury  
Internal Revenue Service 
Cost of Goods Sold
Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065. 
Go to www.irs.gov/Form1125A for the latest information.  
OMB No. 1545-0123
Name
Employer identification number
1
Inventory at beginning of year  .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
1
2
Purchases .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
2
3
Cost of labor 
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
3
4
Additional section 263A costs (attach schedule) .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
4
5
Other costs (attach schedule) 
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
5
6
Total. Add lines 1 through 5 .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
6
7
Inventory at end of year .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

In [18]:
# Define the output file for the cleaned page-level records.
CLEAN_OUTPUT_FILE = OUTPUT_DIR / "irs_pdf_pages_cleaned.jsonl"

# Open the output file in write mode.
with CLEAN_OUTPUT_FILE.open("w", encoding="utf-8") as file:

    # Write one cleaned page record per line.
    for page_record in cleaned_pages:
        file.write(
            json.dumps(
                page_record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Cleaned data saved to:\n{CLEAN_OUTPUT_FILE}")

Cleaned data saved to:
/kaggle/working/irs-extracted-data/irs_pdf_pages_cleaned.jsonl


In [19]:
# Confirm that the cleaned JSONL file was created successfully.
print("File exists:", CLEAN_OUTPUT_FILE.exists())

# Display the file size in bytes.
print("File size:", CLEAN_OUTPUT_FILE.stat().st_size, "bytes")

File exists: True
File size: 1509743 bytes


In [20]:
# Read a few records from the cleaned JSONL file to confirm
# that the saved data can be loaded correctly.
with CLEAN_OUTPUT_FILE.open("r", encoding="utf-8") as file:

    for record_number in range(3):
        line = file.readline()

        if not line:
            break

        record = json.loads(line)

        print(
            f"\nRecord {record_number + 1}: "
            f"{record['document_name']} "
            f"- Page {record['page_number']}"
        )

        print(record["clean_text"][:700])
        print("-" * 80)


Record 1: Form 1125-A (Cost of Goods Sold).pdf - Page 1
Form 1125-A
(Rev. November 2024)
Department of the Treasury
Internal Revenue Service
Cost of Goods Sold
Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065.
Go to www.irs.gov/Form1125A for the latest information.
OMB No. 1545-0123
Name
Employer identification number
1
Inventory at beginning of year .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
1
2
Purchases .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
2
3
Cost of labor
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
3
4
Additional section 263A costs (attach schedule) .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
4
5
Other costs (attach schedule)
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
5
6
Total. Add lines
--------------------------------------------------------------------------------

Record 2: Form 1125-A (Cost of Goods Sold).pdf - Page 2
Form 1125-A (Rev. 11-2024)
Page 2
Specific Instructions
Line 1. Inventory at Beginning of
Year
If you ar

In [21]:
def split_text_into_chunks(
    text: str,
    max_characters: int = 2000,
    overlap_characters: int = 200,
) -> list[str]:
    """
    Split cleaned text into smaller overlapping chunks.

    The overlap helps preserve context when an explanation
    continues from one chunk into the next.

    Parameters
    ----------
    text : str
        Cleaned text from one PDF page.

    max_characters : int
        Maximum approximate size of each chunk.

    overlap_characters : int
        Number of characters repeated between adjacent chunks.

    Returns
    -------
    list[str]
        A list of smaller text chunks.
    """

    # Return an empty list when the page contains no text.
    if not text.strip():
        return []

    # Validate the chunking configuration.
    if overlap_characters >= max_characters:
        raise ValueError(
            "overlap_characters must be smaller than max_characters."
        )

    chunks = []
    start_position = 0
    text_length = len(text)

    while start_position < text_length:

        # Calculate the initial ending position of the chunk.
        end_position = min(
            start_position + max_characters,
            text_length,
        )

        # When possible, end the chunk at the nearest paragraph,
        # line break, or sentence instead of cutting mid-sentence.
        if end_position < text_length:
            possible_endings = [
                text.rfind("\n\n", start_position, end_position),
                text.rfind("\n", start_position, end_position),
                text.rfind(". ", start_position, end_position),
            ]

            best_ending = max(possible_endings)

            # Use the natural boundary only when it is not too
            # close to the beginning of the current chunk.
            if best_ending > start_position + 500:
                end_position = best_ending + 1

        # Extract and store the current chunk.
        chunk_text = text[start_position:end_position].strip()

        if chunk_text:
            chunks.append(chunk_text)

        # Stop when the end of the text has been reached.
        if end_position >= text_length:
            break

        # Move forward while retaining some overlapping context.
        start_position = end_position - overlap_characters

    return chunks

In [22]:
# Store every generated chunk from every cleaned PDF page.
all_chunks = []

for page_record in cleaned_pages:

    # Split the cleaned page text into smaller pieces.
    page_chunks = split_text_into_chunks(
        text=page_record["clean_text"],
        max_characters=2000,
        overlap_characters=200,
    )

    # Preserve the document and page information for each chunk.
    for chunk_index, chunk_text in enumerate(page_chunks, start=1):

        chunk_record = {
            "chunk_id": (
                f"{page_record['document_name']}"
                f"-page-{page_record['page_number']}"
                f"-chunk-{chunk_index}"
            ),
            "document_name": page_record["document_name"],
            "page_number": page_record["page_number"],
            "chunk_number": chunk_index,
            "text": chunk_text,
            "character_count": len(chunk_text),
        }

        all_chunks.append(chunk_record)

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 451


In [23]:
# Display the first few chunks to verify that sentences and
# paragraphs have not been split incorrectly.
for chunk in all_chunks[:3]:

    print("Chunk ID:", chunk["chunk_id"])
    print("Character count:", chunk["character_count"])
    print("-" * 80)
    print(chunk["text"][:1200])
    print("\n" + "=" * 80 + "\n")

Chunk ID: Form 1125-A (Cost of Goods Sold).pdf-page-1-chunk-1
Character count: 1985
--------------------------------------------------------------------------------
Form 1125-A
(Rev. November 2024)
Department of the Treasury
Internal Revenue Service
Cost of Goods Sold
Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065.
Go to www.irs.gov/Form1125A for the latest information.
OMB No. 1545-0123
Name
Employer identification number
1
Inventory at beginning of year .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
1
2
Purchases .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
2
3
Cost of labor
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
3
4
Additional section 263A costs (attach schedule) .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
4
5
Other costs (attach schedule)
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
5
6
Total. Add lines 1 through 5 .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
6
7
Inventory at end of year .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

In [24]:
# Define the output file for the chunk-level dataset.
CHUNK_OUTPUT_FILE = OUTPUT_DIR / "irs_text_chunks.jsonl"

# Open the output file in write mode.
with CHUNK_OUTPUT_FILE.open("w", encoding="utf-8") as file:

    # Write one chunk record per line.
    for chunk_record in all_chunks:
        file.write(
            json.dumps(
                chunk_record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Chunked data saved to:\n{CHUNK_OUTPUT_FILE}")

Chunked data saved to:
/kaggle/working/irs-extracted-data/irs_text_chunks.jsonl


In [25]:
# Confirm that the chunk-level JSONL file exists.
print("File exists:", CHUNK_OUTPUT_FILE.exists())

# Display the size of the file in bytes.
print("File size:", CHUNK_OUTPUT_FILE.stat().st_size, "bytes")

File exists: True
File size: 880156 bytes


In [26]:
# Read a few saved chunk records to confirm that the file
# can be loaded correctly.
with CHUNK_OUTPUT_FILE.open("r", encoding="utf-8") as file:

    for record_number in range(3):
        line = file.readline()

        if not line:
            break

        chunk = json.loads(line)

        print(
            f"\nChunk {record_number + 1}: "
            f"{chunk['chunk_id']}"
        )

        print("Document:", chunk["document_name"])
        print("Page:", chunk["page_number"])
        print("Characters:", chunk["character_count"])

        print("\nText Preview")
        print("-" * 80)
        print(chunk["text"][:800])
        print("=" * 80)


Chunk 1: Form 1125-A (Cost of Goods Sold).pdf-page-1-chunk-1
Document: Form 1125-A (Cost of Goods Sold).pdf
Page: 1
Characters: 1985

Text Preview
--------------------------------------------------------------------------------
Form 1125-A
(Rev. November 2024)
Department of the Treasury
Internal Revenue Service
Cost of Goods Sold
Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065.
Go to www.irs.gov/Form1125A for the latest information.
OMB No. 1545-0123
Name
Employer identification number
1
Inventory at beginning of year .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
1
2
Purchases .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
2
3
Cost of labor
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
3
4
Additional section 263A costs (attach schedule) .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
4
5
Other costs (attach schedule)
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
5
6
Total. Add lines 1 through 5 .
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
6
7
In

In [27]:
# Install Hugging Face Transformers.
#
# This library provides the tokenizer that converts text
# into token IDs understood by the selected base model.
!pip install -q transformers sentencepiece

In [28]:
from transformers import AutoTokenizer


# Define the base model whose tokenizer will be used.
#
# The same tokenizer must later be used during fine-tuning
# and inference.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Download and load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully.
Vocabulary size: 151643


In [29]:
# Create a new list containing the original chunk metadata
# together with the number of tokens in each chunk.
tokenized_chunks = []

for chunk_record in all_chunks:

    # Convert the chunk text into token IDs.
    #
    # Special tokens are excluded here because we are only
    # measuring the source-text length.
    token_ids = tokenizer.encode(
        chunk_record["text"],
        add_special_tokens=False,
    )

    # Copy the original record so it remains unchanged.
    tokenized_record = chunk_record.copy()

    # Store the token count for analysis.
    tokenized_record["token_count"] = len(token_ids)

    tokenized_chunks.append(tokenized_record)

print(f"Token counts calculated for {len(tokenized_chunks)} chunks.")

Token counts calculated for 451 chunks.


In [30]:
# Collect all token counts into one list.
token_counts = [
    chunk["token_count"]
    for chunk in tokenized_chunks
]

print("Number of chunks:", len(token_counts))
print("Minimum tokens:", min(token_counts))
print("Maximum tokens:", max(token_counts))
print("Average tokens:", round(sum(token_counts) / len(token_counts), 2))
print("Total corpus tokens:", sum(token_counts))

Number of chunks: 451
Minimum tokens: 63
Maximum tokens: 892
Average tokens: 462.69
Total corpus tokens: 208674


In [31]:
# Find chunks that exceed 512 tokens.
#
# These may need to be split again before retrieval or training.
oversized_chunks = [
    chunk
    for chunk in tokenized_chunks
    if chunk["token_count"] > 512
]

print("Chunks above 512 tokens:", len(oversized_chunks))

for chunk in oversized_chunks[:10]:
    print(
        chunk["chunk_id"],
        "-",
        chunk["token_count"],
        "tokens",
    )

Chunks above 512 tokens: 155
Form 1125-A (Cost of Goods Sold).pdf-page-1-chunk-1 - 683 tokens
Form 1125-A (Cost of Goods Sold).pdf-page-1-chunk-2 - 519 tokens
Form 1125-A (Cost of Goods Sold).pdf-page-2-chunk-3 - 520 tokens
Instructions for Form 4626.pdf-page-1-chunk-1 - 562 tokens
Instructions for Form 4626.pdf-page-2-chunk-1 - 522 tokens
Instructions for Form 4626.pdf-page-2-chunk-2 - 535 tokens
Instructions for Form 4626.pdf-page-3-chunk-1 - 522 tokens
Instructions for Form 4626.pdf-page-6-chunk-1 - 534 tokens
Instructions for Form 4626.pdf-page-6-chunk-2 - 547 tokens
Instructions for Form 4626.pdf-page-6-chunk-3 - 516 tokens


In [32]:
def split_text_by_tokens(
    text: str,
    tokenizer,
    max_tokens: int = 512,
    overlap_tokens: int = 64,
) -> list[dict]:
    """
    Split text into chunks based on tokenizer token counts.

    This ensures that each chunk remains within the maximum
    token limit supported by the data-processing pipeline.

    Parameters
    ----------
    text : str
        The cleaned source text to split.

    tokenizer
        The tokenizer belonging to the selected base model.

    max_tokens : int
        Maximum number of tokens allowed in one chunk.

    overlap_tokens : int
        Number of tokens repeated between neighboring chunks
        to preserve contextual continuity.

    Returns
    -------
    list[dict]
        A list containing chunk text, token IDs, and token counts.
    """

    # Validate the chunking settings before processing.
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero.")

    if overlap_tokens < 0:
        raise ValueError("overlap_tokens cannot be negative.")

    if overlap_tokens >= max_tokens:
        raise ValueError(
            "overlap_tokens must be smaller than max_tokens."
        )

    # Convert the complete text into token IDs.
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    # Return no chunks when the source text is empty.
    if not token_ids:
        return []

    chunks = []
    start_position = 0

    while start_position < len(token_ids):

        # Select no more than max_tokens for the current chunk.
        end_position = min(
            start_position + max_tokens,
            len(token_ids),
        )

        current_token_ids = token_ids[
            start_position:end_position
        ]

        # Convert the selected token IDs back into readable text.
        chunk_text = tokenizer.decode(
            current_token_ids,
            skip_special_tokens=True,
        ).strip()

        if chunk_text:
            chunks.append(
                {
                    "text": chunk_text,
                    "token_ids": current_token_ids,
                    "token_count": len(current_token_ids),
                }
            )

        # Stop after processing the final group of tokens.
        if end_position >= len(token_ids):
            break

        # Retain overlapping tokens when starting the next chunk.
        start_position = end_position - overlap_tokens

    return chunks

In [33]:
# Store the final token-aware chunk records.
final_chunks = []

for original_chunk in all_chunks:

    # Split each existing character-based chunk according to
    # the selected tokenizer's token count.
    smaller_chunks = split_text_by_tokens(
        text=original_chunk["text"],
        tokenizer=tokenizer,
        max_tokens=512,
        overlap_tokens=64,
    )

    # Create a separate structured record for every resulting
    # token-aware chunk.
    for subchunk_number, subchunk in enumerate(
        smaller_chunks,
        start=1,
    ):
        final_record = {
            "chunk_id": (
                f"{original_chunk['chunk_id']}"
                f"-token-{subchunk_number}"
            ),
            "document_name": original_chunk["document_name"],
            "page_number": original_chunk["page_number"],
            "source_chunk_number": original_chunk["chunk_number"],
            "token_chunk_number": subchunk_number,
            "text": subchunk["text"],
            "character_count": len(subchunk["text"]),
            "token_count": subchunk["token_count"],
        }

        final_chunks.append(final_record)

print(f"Original chunks: {len(all_chunks)}")
print(f"Final token-aware chunks: {len(final_chunks)}")

Original chunks: 451
Final token-aware chunks: 606


In [34]:
# Collect the final token counts for validation.
final_token_counts = [
    chunk["token_count"]
    for chunk in final_chunks
]

print("Minimum tokens:", min(final_token_counts))
print("Maximum tokens:", max(final_token_counts))
print(
    "Average tokens:",
    round(
        sum(final_token_counts) / len(final_token_counts),
        2,
    ),
)
print("Total tokens:", sum(final_token_counts))

# Confirm that no chunk exceeds the selected limit.
invalid_chunks = [
    chunk
    for chunk in final_chunks
    if chunk["token_count"] > 512
]

print("Chunks exceeding 512 tokens:", len(invalid_chunks))

Minimum tokens: 63
Maximum tokens: 512
Average tokens: 360.72
Total tokens: 218594
Chunks exceeding 512 tokens: 0


## Define the Knowledge Record Schema

In [35]:
# Install Pydantic for defining and validating structured data.
#
# Pydantic ensures that every knowledge record follows the
# same format and contains valid values.
!pip install -q pydantic

In [36]:
# Import Optional because some fields, such as schedule or line,
# may not apply to every IRS instruction section.
from typing import Optional

# Import BaseModel and Field for building the validation schema.
from pydantic import BaseModel, Field

In [37]:
class Form1120KnowledgeRecord(BaseModel):
    """
    Represents one structured knowledge record derived from
    an official IRS Form 1120 document.

    A record may describe a form line, schedule, section,
    filing instruction, definition, exception, or requirement.
    """

    # Unique identifier used to locate and reference the record.
    record_id: str = Field(
        ...,
        min_length=1,
        description="Unique identifier for the knowledge record.",
    )

    # Tax year to which the instruction applies.
    tax_year: int = Field(
        ...,
        ge=2000,
        description="Tax year covered by the IRS source.",
    )

    # Main IRS form associated with the record.
    form: str = Field(
        ...,
        min_length=1,
        description="IRS form number, such as 1120.",
    )

    # Name of the original source PDF.
    document_name: str = Field(
        ...,
        min_length=1,
        description="Filename of the official IRS source document.",
    )

    # Page number in the source PDF.
    source_page: int = Field(
        ...,
        ge=1,
        description="Page number where the source text appears.",
    )

    # Schedule name when the record belongs to a schedule.
    schedule: Optional[str] = Field(
        default=None,
        description="Schedule name, such as Schedule C or Schedule K.",
    )

    # Major subject area in the source document.
    section: Optional[str] = Field(
        default=None,
        description="Major section, such as Income or Deductions.",
    )

    # More specific heading contained within a section.
    subsection: Optional[str] = Field(
        default=None,
        description="Subsection or instruction heading.",
    )

    # Form line identifier, such as 1a, 3, or 26.
    line: Optional[str] = Field(
        default=None,
        description="Form or schedule line associated with the record.",
    )

    # Human-readable title of the instruction.
    title: str = Field(
        ...,
        min_length=1,
        description="Title or heading of the knowledge record.",
    )

    # Cleaned instruction text taken from the IRS source.
    instruction: str = Field(
        ...,
        min_length=1,
        description="Official instruction text associated with the record.",
    )

    # Search terms that can help retrieve this record.
    keywords: list[str] = Field(
        default_factory=list,
        description="Keywords used for search and retrieval.",
    )

    # Other IRS forms or schedules referenced by the instruction.
    related_forms: list[str] = Field(
        default_factory=list,
        description="Related IRS forms and schedules.",
    )

    # Identifier of the source text chunk used to create the record.
    source_chunk_id: str = Field(
        ...,
        min_length=1,
        description="Identifier of the original extracted text chunk.",
    )

In [38]:
# Create one example record to verify that the schema works.
#
# This is only a test record. Do not begin creating the full
# Form 1120 knowledge dataset yet.
sample_record = Form1120KnowledgeRecord(
    record_id="2025-1120-P1-L3",
    tax_year=2025,
    form="1120",
    document_name="i1120.pdf",
    source_page=8,
    schedule=None,
    section="Income",
    subsection="Specific Instructions",
    line="3",
    title="Gross profit",
    instruction=(
        "Gross profit is determined by subtracting line 2 "
        "from line 1c."
    ),
    keywords=[
        "gross profit",
        "income",
        "line 3",
        "cost of goods sold",
    ],
    related_forms=[
        "Form 1125-A",
    ],
    source_chunk_id="i1120.pdf-page-8-chunk-2",
)

print(sample_record)

record_id='2025-1120-P1-L3' tax_year=2025 form='1120' document_name='i1120.pdf' source_page=8 schedule=None section='Income' subsection='Specific Instructions' line='3' title='Gross profit' instruction='Gross profit is determined by subtracting line 2 from line 1c.' keywords=['gross profit', 'income', 'line 3', 'cost of goods sold'] related_forms=['Form 1125-A'] source_chunk_id='i1120.pdf-page-8-chunk-2'


In [39]:
# Convert the validated Pydantic object into formatted JSON.
#
# This shows how each final knowledge record will be stored
# in the dataset.
print(
    sample_record.model_dump_json(
        indent=2,
    )
)

{
  "record_id": "2025-1120-P1-L3",
  "tax_year": 2025,
  "form": "1120",
  "document_name": "i1120.pdf",
  "source_page": 8,
  "schedule": null,
  "section": "Income",
  "subsection": "Specific Instructions",
  "line": "3",
  "title": "Gross profit",
  "instruction": "Gross profit is determined by subtracting line 2 from line 1c.",
  "keywords": [
    "gross profit",
    "income",
    "line 3",
    "cost of goods sold"
  ],
  "related_forms": [
    "Form 1125-A"
  ],
  "source_chunk_id": "i1120.pdf-page-8-chunk-2"
}


In [40]:
from pathlib import Path

# Define the directory where schema files will be stored.
SCHEMA_DIR = Path(
    "/kaggle/working/form-1120-project/schemas"
)

# Create the directory and any missing parent directories.
SCHEMA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Schema directory:\n{SCHEMA_DIR}")

Schema directory:
/kaggle/working/form-1120-project/schemas


In [41]:
# Generate a JSON-compatible description of the
# Form1120KnowledgeRecord model.
#
# The generated schema describes:
# - Required fields
# - Optional fields
# - Data types
# - Validation limits
# - Field descriptions
knowledge_json_schema = (
    Form1120KnowledgeRecord.model_json_schema()
)

print("JSON schema generated successfully.")

JSON schema generated successfully.


In [42]:
import json

# Define the complete output path for the schema file.
SCHEMA_OUTPUT_FILE = (
    SCHEMA_DIR /
    "form_1120_knowledge_record_schema.json"
)

# Save the schema as formatted JSON so that it remains
# readable for developers, reviewers, and future systems.
with SCHEMA_OUTPUT_FILE.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        knowledge_json_schema,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Knowledge schema saved to:\n"
    f"{SCHEMA_OUTPUT_FILE}"
)

Knowledge schema saved to:
/kaggle/working/form-1120-project/schemas/form_1120_knowledge_record_schema.json


In [43]:
# Confirm that the schema file was created successfully.
print(
    "File exists:",
    SCHEMA_OUTPUT_FILE.exists(),
)

# Display the size of the schema file.
print(
    "File size:",
    SCHEMA_OUTPUT_FILE.stat().st_size,
    "bytes",
)

File exists: True
File size: 3232 bytes


In [44]:
# Load the saved schema back into Python to verify that
# the JSON file is valid and readable.
with SCHEMA_OUTPUT_FILE.open(
    "r",
    encoding="utf-8",
) as file:
    saved_schema = json.load(file)

# Display the schema in a readable format.
print(
    json.dumps(
        saved_schema,
        ensure_ascii=False,
        indent=2,
    )[:5000]
)

{
  "description": "Represents one structured knowledge record derived from\nan official IRS Form 1120 document.\n\nA record may describe a form line, schedule, section,\nfiling instruction, definition, exception, or requirement.",
  "properties": {
    "record_id": {
      "description": "Unique identifier for the knowledge record.",
      "minLength": 1,
      "title": "Record Id",
      "type": "string"
    },
    "tax_year": {
      "description": "Tax year covered by the IRS source.",
      "minimum": 2000,
      "title": "Tax Year",
      "type": "integer"
    },
    "form": {
      "description": "IRS form number, such as 1120.",
      "minLength": 1,
      "title": "Form",
      "type": "string"
    },
    "document_name": {
      "description": "Filename of the official IRS source document.",
      "minLength": 1,
      "title": "Document Name",
      "type": "string"
    },
    "source_page": {
      "description": "Page number where the source text appears.",
      "minimum"

In [45]:
from pathlib import Path
import json

# Point to the token-aware chunk dataset created earlier.
#
# Update this path only if you saved the file under a
# different directory or filename.
TOKEN_CHUNK_FILE = Path(
    "/kaggle/working/form-1120-project/data/"
    "irs_text_chunks_token_aware.jsonl"
)

print(f"Chunk dataset path:\n{TOKEN_CHUNK_FILE}")

Chunk dataset path:
/kaggle/working/form-1120-project/data/irs_text_chunks_token_aware.jsonl


In [46]:
from pathlib import Path

TOKEN_CHUNK_FILE = Path(
    "/kaggle/working/form-1120-project/data/"
    "irs_text_chunks_token_aware.jsonl"
)

print("File exists:", TOKEN_CHUNK_FILE.exists())

File exists: False


In [47]:
from pathlib import Path
import json

CHUNK_FILE = Path(
    "/kaggle/working/irs-extracted-data/"
    "irs_text_chunks.jsonl"
)

print("File exists:", CHUNK_FILE.exists())

with CHUNK_FILE.open("r", encoding="utf-8") as file:
    first_record = json.loads(file.readline())

print(
    json.dumps(
        first_record,
        indent=2,
        ensure_ascii=False,
    )
)

File exists: True
{
  "chunk_id": "Form 1125-A (Cost of Goods Sold).pdf-page-1-chunk-1",
  "document_name": "Form 1125-A (Cost of Goods Sold).pdf",
  "page_number": 1,
  "chunk_number": 1,
  "text": "Form 1125-A\n(Rev. November 2024)\nDepartment of the Treasury\nInternal Revenue Service\nCost of Goods Sold\nAttach to Form 1120, 1120-C, 1120-F, 1120S, or 1065.\nGo to www.irs.gov/Form1125A for the latest information.\nOMB No. 1545-0123\nName\nEmployer identification number\n1\nInventory at beginning of year .\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n1\n2\nPurchases .\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n2\n3\nCost of labor\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n3\n4\nAdditional section 263A costs (attach schedule) .\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n4\n5\nOther costs (attach schedule)\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.\n.

In [48]:
# Load every chunk from the JSONL dataset.
#
# Each line in a JSONL file contains one independent JSON object.
# Blank lines are ignored to prevent parsing errors.

all_chunks = []

with CHUNK_FILE.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):

        # Skip empty lines.
        if not line.strip():
            continue

        try:
            chunk = json.loads(line)

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number}: {error}"
            ) from error

        all_chunks.append(chunk)

print("Total chunks loaded:", len(all_chunks))

Total chunks loaded: 451


In [49]:
 # Import Python's regular expression module.
# We'll use regex to identify IRS line numbers in the text.
import re

In [50]:
def extract_line_numbers(text: str) -> list[str]:
    """
    Extract IRS line numbers from a block of text.

    Examples:
        1
        1a
        9d
        9d(i)
        26
    """

    pattern = (
        r"\b\d+[a-z]?(?:\([ivx]+\))?\b"
    )

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE,
    )

    # Remove duplicates while preserving order.
    return list(dict.fromkeys(matches))

In [51]:
line_numbers = extract_line_numbers(
    first_record["text"]
)

print("Detected line numbers:")
print(line_numbers)

Detected line numbers:
['1125', '2024', '1120', '1120S', '1065', '1545', '0123', '1', '2', '3', '4', '263A', '5', '6', '7', '8', '9a', '970', '9d']


In [52]:
# Show each detected line number on a separate line.
#
# This makes it easier to spot false matches such as:
# - years
# - form numbers
# - OMB numbers
# - section references
# - repeated line labels

for line_number in line_numbers:
    print(line_number)

1125
2024
1120
1120S
1065
1545
0123
1
2
3
4
263A
5
6
7
8
9a
970
9d


In [53]:
# Detect IRS line labels that appear at the start of a line.
#
# This reduces false matches from:
# - tax years such as 2024
# - form numbers such as 1120
# - section references such as 263A
# - OMB numbers

def extract_starting_line_labels(text: str) -> list[str]:
    """
    Extract likely IRS form line labels from the beginning
    of individual text lines.

    Examples:
        1
        9a
        9d(i)
        26
    """

    detected_labels = []

    for raw_line in text.splitlines():
        line = raw_line.strip()

        match = re.match(
            r"^(\d{1,2}[a-z]?(?:\([ivx]+\))?)\b",
            line,
            flags=re.IGNORECASE,
        )

        if match:
            detected_labels.append(match.group(1))

    # Remove duplicates while preserving their original order.
    return list(dict.fromkeys(detected_labels))


starting_line_labels = extract_starting_line_labels(
    first_record["text"]
)

print("Detected starting line labels:")
print(starting_line_labels)

Detected starting line labels:
['1', '2', '3', '4', '5', '6', '7', '8', '9a', '9d']


In [54]:
# Extract the IRS form number from a document filename.
#
# Examples:
# "Form 1125-A (Cost of Goods Sold).pdf" -> "1125-A"
# "Form 1120.pdf" -> "1120"

def extract_form_number(document_name: str) -> str | None:
    """
    Extract an IRS form number from a document filename.

    Returns None when no recognizable form number is found.
    """

    match = re.search(
        r"\bForm\s+([0-9]{3,4}(?:-[A-Z])?)\b",
        document_name,
        flags=re.IGNORECASE,
    )

    if match:
        return match.group(1).upper()

    return None


detected_form = extract_form_number(
    first_record["document_name"]
)

print("Detected form:", detected_form)

Detected form: 1125-A


In [55]:
# Extract the tax year from the document text.
#
# Examples of text:
#   "2025 Instructions for Form 1120"
#   "(Rev. November 2024)"
#
# Returns the first reasonable year found.

def extract_tax_year(text: str) -> int | None:
    """
    Extract the tax year from IRS document text.

    Returns
    -------
    int | None
        Detected year, or None if no year is found.
    """

    match = re.search(
        r"\b(20\d{2})\b",
        text,
    )

    if match:
        return int(match.group(1))

    return None

In [56]:
# Test the Function

detected_tax_year = extract_tax_year(
    first_record["text"]
)

print("Detected tax year:", detected_tax_year)

Detected tax year: 2024


In [57]:
# -----------------------------------------------------------------------------
# Function: extract_document_type
#
# Purpose
# -------
# Determine the type of IRS document based on its filename.
#
# Why This Function Exists
# ------------------------
# The Form 1120 project contains multiple document categories, including
# official IRS forms, instruction booklets, schedules, and supporting
# documents.
#
# Identifying the document type allows later stages of the pipeline to apply
# different parsing strategies depending on the source document.
#
# Example Results
# ---------------
# "Form 1125-A (Cost of Goods Sold).pdf"
#     -> Form
#
# "Instructions for Form 1120.pdf"
#     -> Instructions
#
# "Schedule M-3.pdf"
#     -> Schedule
#
# Returns
# -------
# str
#     The detected document type.
# -----------------------------------------------------------------------------

def extract_document_type(document_name: str) -> str:
    """
    Determine the IRS document type from its filename.
    """

    document_name = document_name.lower()

    # Check whether the filename represents an instruction booklet.
    if "instruction" in document_name:
        return "Instructions"

    # Check whether the filename represents a schedule.
    if "schedule" in document_name:
        return "Schedule"

    # Check whether the filename represents an IRS form.
    if "form" in document_name:
        return "Form"

    # Return "Unknown" when the filename does not match any known pattern.
    return "Unknown"

In [58]:
# -----------------------------------------------------------------------------
# Test the document type extraction function.
#
# Purpose
# -------
# Verify that the parser correctly classifies the current IRS document.
#
# Expected Result
# ---------------
# For the first record:
#
#     Form
# -----------------------------------------------------------------------------

document_type = extract_document_type(
    first_record["document_name"]
)

print("Detected document type:", document_type)

Detected document type: Form


In [59]:
# -----------------------------------------------------------------------------
# Function: extract_document_title
#
# Purpose
# -------
# Extract a readable title from the beginning of an IRS document.
#
# Why This Function Exists
# ------------------------
# The filename is useful for locating the source file, but a readable document
# title is better for:
#
# - knowledge records
# - search results
# - citations
# - dataset review
# - user-facing explanations
#
# For example, this function should identify:
#
#     "Cost of Goods Sold"
#
# from the first page of Form 1125-A.
#
# How It Works
# ------------
# 1. Splits the extracted text into individual lines.
# 2. Removes blank lines.
# 3. Ignores common administrative headings such as:
#    - Department of the Treasury
#    - Internal Revenue Service
#    - revision dates
#    - form numbers
# 4. Returns the first suitable descriptive line.
#
# Important Note
# --------------
# This is an initial rule-based version. IRS layouts vary, so the function
# may be refined after testing it across all source documents.
# -----------------------------------------------------------------------------

def extract_document_title(text: str) -> str | None:
    """
    Extract a readable IRS document title from the document text.

    Parameters
    ----------
    text : str
        Extracted text from the first page of an IRS document.

    Returns
    -------
    str | None
        The detected document title, or None if no suitable title is found.
    """

    # Convert the text into separate lines and remove surrounding whitespace.
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Words and phrases that usually represent administrative information
    # rather than the actual document title.
    ignored_patterns = [
        r"^Form\s+\d",
        r"^\(Rev\.",
        r"^Department of the Treasury$",
        r"^Internal Revenue Service$",
        r"^OMB No\.",
        r"^Attach to Form",
        r"^Go to www\.",
        r"^Name$",
        r"^Employer identification number$",
    ]

    for line in lines:

        # Determine whether the current line matches an administrative pattern.
        should_ignore = any(
            re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            )
            for pattern in ignored_patterns
        )

        # Skip administrative or non-title lines.
        if should_ignore:
            continue

        # Ignore very short values such as isolated line numbers.
        if len(line) < 4:
            continue

        # Return the first line that looks like a meaningful document title.
        return line

    # Return None if no suitable title can be identified.
    return None

In [60]:
# -----------------------------------------------------------------------------
# Test the document title extraction function.
#
# Purpose
# -------
# Confirm that the first record produces a meaningful title before applying
# the function to the complete dataset.
#
# Expected Result
# ---------------
# For Form 1125-A, the expected title is:
#
#     Cost of Goods Sold
# -----------------------------------------------------------------------------

document_title = extract_document_title(
    first_record["text"]
)

print("Detected document title:", document_title)

Detected document title: Cost of Goods Sold


In [61]:
# -----------------------------------------------------------------------------
# Function: extract_schedule_name
#
# Purpose
# -------
# Detect an IRS schedule name from either the document filename or the
# extracted document text.
#
# Why This Function Exists
# ------------------------
# Some knowledge records belong to schedules rather than the main form.
# Storing the schedule name makes later search and retrieval more precise.
#
# Example Results
# ---------------
# "Schedule M-3 (Form 1120).pdf" -> "Schedule M-3"
# "Schedule C"                  -> "Schedule C"
# "Form 1125-A.pdf"             -> None
#
# Detection Order
# ---------------
# 1. Search the filename first because it is usually cleaner.
# 2. Search the extracted text only if the filename does not contain a
#    schedule name.
# 3. Return None when no schedule is found.
# -----------------------------------------------------------------------------

def extract_schedule_name(
    document_name: str,
    text: str,
) -> str | None:
    """
    Extract an IRS schedule name from a filename or text.

    Parameters
    ----------
    document_name : str
        Name of the source PDF.

    text : str
        Extracted text from the document.

    Returns
    -------
    str | None
        Detected schedule name, or None when the document is not a schedule.
    """

    # Match schedule labels such as:
    # Schedule C
    # Schedule K
    # Schedule M-3
    schedule_pattern = r"\bSchedule\s+([A-Z](?:-\d+)?)\b"

    # Search the filename first because filenames are usually less noisy.
    filename_match = re.search(
        schedule_pattern,
        document_name,
        flags=re.IGNORECASE,
    )

    if filename_match:
        schedule_code = filename_match.group(1).upper()
        return f"Schedule {schedule_code}"

    # Search the extracted text when the filename does not identify a schedule.
    text_match = re.search(
        schedule_pattern,
        text,
        flags=re.IGNORECASE,
    )

    if text_match:
        schedule_code = text_match.group(1).upper()
        return f"Schedule {schedule_code}"

    # Return None for documents such as Form 1125-A that are not schedules.
    return None

In [62]:
# -----------------------------------------------------------------------------
# Test the schedule extraction function on the first record.
#
# Expected Result
# ---------------
# The first record belongs to Form 1125-A, not an IRS schedule.
#
# Expected output:
#     Detected schedule: None
# -----------------------------------------------------------------------------

detected_schedule = extract_schedule_name(
    document_name=first_record["document_name"],
    text=first_record["text"],
)

print("Detected schedule:", detected_schedule)

Detected schedule: None


In [63]:
# -----------------------------------------------------------------------------
# Function: extract_part_labels
#
# Purpose
# -------
# Detect IRS document part labels from a chunk of extracted text.
#
# Why This Function Exists
# ------------------------
# Many IRS forms and schedules organize content into numbered parts, such as:
#
#     Part I
#     Part II
#     Part III
#
# Capturing these labels helps preserve the document hierarchy and makes the
# final knowledge records easier to search, group, and cite.
#
# How It Works
# ------------
# 1. Searches the text for the word "Part".
# 2. Accepts Roman numerals such as I, II, III, IV, V, and X.
# 3. Removes duplicates while preserving the original order.
#
# Important Note
# --------------
# This function only detects the part label. It does not yet determine which
# later lines belong to that part. That association will be added after the
# basic detectors are tested.
# -----------------------------------------------------------------------------

def extract_part_labels(text: str) -> list[str]:
    """
    Extract IRS part labels from a block of text.

    Parameters
    ----------
    text : str
        Extracted text from one IRS document chunk.

    Returns
    -------
    list[str]
        Unique part labels in the order they appear.
    """

    # Match labels such as:
    # Part I
    # Part II
    # Part IV
    # Part X
    matches = re.findall(
        r"\bPart\s+([IVX]+)\b",
        text,
        flags=re.IGNORECASE,
    )

    # Rebuild each result in a consistent format.
    normalized_parts = [
        f"Part {roman_numeral.upper()}"
        for roman_numeral in matches
    ]

    # Remove duplicate labels while preserving their original order.
    return list(dict.fromkeys(normalized_parts))

In [64]:
# -----------------------------------------------------------------------------
# Test the part-label detector using the first chunk.
#
# Expected Result
# ---------------
# Form 1125-A does not normally contain labels such as "Part I" in the first
# extracted chunk, so the expected result may be an empty list:
#
#     []
# -----------------------------------------------------------------------------

detected_parts = extract_part_labels(
    first_record["text"]
)

print("Detected parts:", detected_parts)

Detected parts: []


In [65]:
# -----------------------------------------------------------------------------
# Function: extract_section_headings
#
# Purpose
# -------
# Identify major section headings within IRS documents.
#
# Why This Function Exists
# ------------------------
# IRS documents are organized into logical sections such as:
#
# • Income
# • Deductions
# • Specific Instructions
# • General Instructions
# • Accounting Methods
#
# Capturing these headings allows the knowledge base to organize
# instructions hierarchically instead of treating the document as
# plain text.
#
# How It Works
# ------------
# 1. Examines each line of text individually.
# 2. Ignores blank lines.
# 3. Selects lines that appear to be headings.
# 4. Removes duplicate headings while preserving order.
#
# Important Note
# --------------
# This is Version 1 of the detector.
# Later versions will use additional formatting and contextual rules
# to improve heading detection.
# -----------------------------------------------------------------------------

def extract_section_headings(text: str) -> list[str]:
    """
    Extract likely section headings from IRS document text.

    Parameters
    ----------
    text : str
        Extracted text from one IRS document chunk.

    Returns
    -------
    list[str]
        Candidate section headings.
    """

    headings = []

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        # Ignore lines that are mostly numbers.
        if re.fullmatch(r"[\d().A-Za-z-]+", line) and len(line) <= 3:
            continue

        # Ignore administrative lines.
        if line.lower().startswith("department"):
            continue

        if line.lower().startswith("internal revenue"):
            continue

        if line.lower().startswith("omb"):
            continue

        # Candidate heading:
        # Short descriptive line with no period.
        if (
            len(line) <= 60
            and "." not in line
            and len(line.split()) <= 8
        ):
            headings.append(line)

    # Remove duplicates while preserving order.
    return list(dict.fromkeys(headings))

In [66]:
# -----------------------------------------------------------------------------
# Test the section heading detector.
#
# Purpose
# -------
# Display all candidate headings detected from the first chunk.
#
# This allows us to evaluate whether the heuristic is identifying
# meaningful section titles or capturing too much noise.
# -----------------------------------------------------------------------------

section_headings = extract_section_headings(
    first_record["text"]
)

print("Detected section headings:\n")

for heading in section_headings:
    print("-", heading)

Detected section headings:

- Form 1125-A
- Cost of Goods Sold
- Name
- Employer identification number
- Cost of labor
- Other costs (attach schedule)
- Cost
- (ii)
- Lower of cost or market
- (iii)
- Other (specify method used and attach explanation)
- (iv)
- Non-incidental materials and supplies method
- AFS method
- (vi)
- Non-AFS method
- 9d(i)
- 9d(ii)


In [67]:
# -----------------------------------------------------------------------------
# Purpose
# -------
# Convert the first chunk into a numbered list of non-empty text lines.
#
# Why This Step Exists
# --------------------
# PDF extraction does not always preserve the visual layout of an IRS form.
# A line number, field title, and description may appear on separate lines.
#
# Viewing indexed lines helps us understand the actual extraction pattern
# before writing the line-instruction parser.
# -----------------------------------------------------------------------------

indexed_text_lines = []

# Split the first chunk into individual lines.
for original_line_number, raw_line in enumerate(
    first_record["text"].splitlines(),
    start=1,
):
    # Remove whitespace around the extracted line.
    cleaned_line = raw_line.strip()

    # Ignore empty lines because they do not contain useful content.
    if not cleaned_line:
        continue

    # Store both the original position and cleaned text.
    indexed_text_lines.append(
        {
            "original_line_number": original_line_number,
            "text": cleaned_line,
        }
    )

# Display the indexed lines in a readable format.
for item in indexed_text_lines:
    print(
        f"{item['original_line_number']:>4}: "
        f"{item['text']}"
    )

   1: Form 1125-A
   2: (Rev. November 2024)
   3: Department of the Treasury
   4: Internal Revenue Service
   5: Cost of Goods Sold
   6: Attach to Form 1120, 1120-C, 1120-F, 1120S, or 1065.
   7: Go to www.irs.gov/Form1125A for the latest information.
   8: OMB No. 1545-0123
   9: Name
  10: Employer identification number
  11: 1
  12: Inventory at beginning of year .
  13: .
  14: .
  15: .
  16: .
  17: .
  18: .
  19: .
  20: .
  21: .
  22: .
  23: .
  24: .
  25: .
  26: .
  27: .
  28: .
  29: .
  30: .
  31: .
  32: .
  33: .
  34: .
  35: .
  36: .
  37: 1
  38: 2
  39: Purchases .
  40: .
  41: .
  42: .
  43: .
  44: .
  45: .
  46: .
  47: .
  48: .
  49: .
  50: .
  51: .
  52: .
  53: .
  54: .
  55: .
  56: .
  57: .
  58: .
  59: .
  60: .
  61: .
  62: .
  63: .
  64: .
  65: .
  66: .
  67: .
  68: .
  69: .
  70: 2
  71: 3
  72: Cost of labor
  73: .
  74: .
  75: .
  76: .
  77: .
  78: .
  79: .
  80: .
  81: .
  82: .
  83: .
  84: .
  85: .
  86: .
  87: .
  88

In [68]:
# -----------------------------------------------------------------------------
# Function: extract_basic_line_records
#
# Purpose
# -------
# Convert extracted IRS form text into simple line-level records.
#
# Why This Function Exists
# ------------------------
# In the PDF extraction, form line numbers and their descriptions often appear
# on separate lines. For example:
#
#     1
#     Inventory at beginning of year
#
# This function detects the numeric line label and captures the next meaningful
# text line as its description.
#
# Important Limitation
# --------------------
# This is an initial parser. It does not yet handle every complex case such as:
#
# - sub-lines like 9a, 9b, or 9d(i)
# - descriptions spanning multiple lines
# - repeated line numbers at the right side of a form
# - dotted leader characters
#
# Those cases will be added after this basic version is tested.
# -----------------------------------------------------------------------------

def extract_basic_line_records(text: str) -> list[dict]:
    """
    Extract basic IRS line-number and description pairs.

    Parameters
    ----------
    text : str
        Extracted text from one IRS form chunk.

    Returns
    -------
    list[dict]
        A list of dictionaries containing:
        - line
        - description
    """

    # Split the chunk into separate lines and remove blank lines.
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    records = []

    # Examine each line together with the line immediately following it.
    for index in range(len(lines) - 1):

        current_line = lines[index]
        next_line = lines[index + 1]

        # Match simple line labels such as:
        # 1
        # 2
        # 9a
        # 9d(i)
        line_match = re.fullmatch(
            r"\d{1,2}[a-z]?(?:\([ivx]+\))?",
            current_line,
            flags=re.IGNORECASE,
        )

        if not line_match:
            continue

        # Ignore the next value when it is only punctuation or another number.
        if re.fullmatch(r"[.\d()]+", next_line):
            continue

        records.append(
            {
                "line": current_line,
                "description": next_line,
            }
        )

    return records

In [69]:
# -----------------------------------------------------------------------------
# Test the basic line-record extractor on the first chunk.
#
# Purpose
# -------
# Confirm that the parser can identify simple pairs such as:
#
#     1 -> Inventory at beginning of year
#     2 -> Purchases
#     3 -> Cost of labor
#
# The output may still contain imperfect matches. That is expected at this
# stage because this is only the first parser version.
# -----------------------------------------------------------------------------

basic_line_records = extract_basic_line_records(
    first_record["text"]
)

print("Detected basic line records:\n")

for record in basic_line_records:
    print(
        f"Line {record['line']}: "
        f"{record['description']}"
    )

Detected basic line records:

Line 1: Inventory at beginning of year .
Line 2: Purchases .
Line 3: Cost of labor
Line 4: Additional section 263A costs (attach schedule) .
Line 5: Other costs (attach schedule)
Line 6: Total. Add lines 1 through 5 .
Line 7: Inventory at end of year .
Line 8: Cost of goods sold. Subtract line 7 from line 6. Enter here and on Form 1120, page 1, line 2, or the appropriate
Line 8: 9a
Line 9a: Check all methods used for valuing closing inventory. See instructions.
Line 9d(i): (ii)
Line 9d(ii): e


In [70]:
# -----------------------------------------------------------------------------
# Function: build_candidate_knowledge_record
#
# Purpose
# -------
# Build a candidate Form 1120 knowledge record from one extracted line.
#
# Why This Function Exists
# ------------------------
# Up to this point, we've extracted individual pieces of information:
#
# • Form number
# • Tax year
# • Document type
# • Document title
# • Schedule name
# • Line number
# • Line description
#
# This function combines all of those pieces into a single structured record
# that matches our knowledge schema.
#
# Important Note
# --------------
# This creates a candidate record only. The record will later be validated
# using the Pydantic schema before being saved.
# -----------------------------------------------------------------------------

def build_candidate_knowledge_record(
    first_record: dict,
    line_record: dict,
) -> dict:
    """
    Build one candidate knowledge record.

    Parameters
    ----------
    first_record : dict
        Original chunk record.

    line_record : dict
        Extracted line information.

    Returns
    -------
    dict
        Candidate knowledge record.
    """

    return {
        "record_id": (
            f"{extract_form_number(first_record['document_name'])}"
            f"-P{first_record['page_number']}"
            f"-L{line_record['line']}"
        ),

        "tax_year": extract_tax_year(
            first_record["text"]
        ),

        "form": extract_form_number(
            first_record["document_name"]
        ),

        "document_name": first_record["document_name"],

        "source_page": first_record["page_number"],

        "schedule": extract_schedule_name(
            first_record["document_name"],
            first_record["text"],
        ),

        "section": None,

        "subsection": None,

        "line": line_record["line"],

        "title": line_record["description"],

        "instruction": line_record["description"],

        "keywords": [],

        "related_forms": [],

        "source_chunk_id": first_record["chunk_id"],
    }